In [1]:
class Transformer:

    def __init__(self, num_dims, num_heads):
        self.ln_1 = LayerNorm(num_dims)
        self.attn = CausalSelfAttention(num_dims, num_heads)
        self.ln_2 = LayerNorm(num_dims)
        self.mlp = MyMlp(in_features=num_dims, hidden_features=4 * num_dims, out_features=num_dims)
        
    def forward(self, x):
        norm1 = self.ln_1.forward(x)
        attn_out = self.attn.forward(norm1)
        x1 = x + attn_out

        norm2 = self.ln_2.forward(x1)
        mlp_out = self.mlp.forward(norm2)
        x2 = x1 + mlp_out

        return x2

    def backward(self, grad_out):
        
        grad_mlp = self.mlp.backward(grad_out)
        grad_norm2 = self.ln_2.backward(grad_mlp)
        grad_x1 = grad_out + grad_norm2

        grad_attn = self.attn.backward(grad_x1)
        grad_norm1 = self.ln_1.backward(grad_attn)
        grad_x = grad_x1 + grad_norm1

        return grad_x

```
The below explanation is from gemini,cause it did explained better than my raw intuition
```
### Transformer Block: The Real Deal

Think of this block as the engine that turns dumb, isolated tokens into tokens that actually know what’s going on around them.

When a token enters, it only knows its own ID. By the time it leaves this block, it knows who came before it, what those words meant, and what fact it needs to pull up.

#### The 4x Trick in MLP: Why Expand and Squeeze?

Why not just keep everything at `d_model`?
Because if you keep the dimension tight, the model doesn't have enough elbow room to separate complicated ideas.

* We blow it up to $4 \times d_{\text{model}}$ so the activation (ReLU/GELU) can cut through high-dimensional space and untangle messy features.
* Then we slam it back down to $d_{\text{model}}$ to keep only the good stuff and throw away the noise before putting it back on the highway.

#### Why Pre-LN and Not Post-LN?

* **Post-LN (`LN(x + SubLayer(x))`):** This was the old way. The problem? Every time you normalize right after adding, you mess with the running highway. Stack 12 or 24 blocks of that, and by the time your backward pass hits the bottom layers, the gradients are completely wiped out. You end up babying the model with tiny warmups just so it doesn't blow up.
* **Pre-LN (`x + SubLayer(LN(x))`):** This is what modern GPTs use. You leave the main highway $x$ alone. You only normalize the branch that goes into Attention or MLP. Because the main road is pure addition ($x + \dots$), its derivative is literally $1$. Gradients can fly straight from the loss down to the very first token embedding without dying.

#### Forward Pass

1. Grab $x$ from the highway.
2. Hit it with `ln_1`, throw that into Attention.
3. Add attention's output back to our original $x$ $\to$ this is `x1`.
4. Hit `x1` with `ln_2`, throw that into MLP (expands $4\times$, activates, squeezes back).
5. Add MLP's output back to `x1` $\to$ this is `x2`. Done.

#### Backward Pass

No magic math here. It's just like the MLP backward—you run the exact reverse order. The only rule to keep in mind: **every time we added in forward, the gradient splits in backward.**

1. `grad_out` hits the block.
2. Route it back through `mlp` then `ln_2`.
3. Add the skip gradient: `grad_x1 = grad_out + grad_norm2`.
4. Route `grad_x1` through `attn` then `ln_1`.
5. Add the skip gradient: `grad_x = grad_x1 + grad_norm1`.
6. Return `grad_x` to the guy behind us.